# Soter v-final: T4 execution package
**NOT RUN on pretrained models. Experimental, not a released model.** Upload this repository's `vfinal_pipeline`, `eval_harness`, `configs/v-final`, and docs to the Drive path below. Stage accepted, licensed, privacy-reviewed JSONL sources and a disjoint frozen holdout. Do not upload raw production PII. No credential collection. Select a GPU runtime. Cells use real local-only training/export code; downloads and installs require explicit flags.
Native tests used tiny randomly initialized models only, not learned Soter output. Colab pinned dependencies were metadata-checked, not installed/tested here. Native versions differ. Read `docs/ml/v-final-offline/execution/RUNBOOK.md` before enabling training. Long-context truncation and unresolved product whitelist bugs remain release blockers.

In [ ]:
import os, sys, json, copy, time, subprocess
from pathlib import Path
RUN_INSTALL = False
RUN_DOWNLOAD = False
RUN_TRAIN = False
RUN_ABLATIONS = False
RUN_EXPORT = False
RUN_TEACHER_TRAIN = False
RUN_GATE = False
RUN_PACKAGE = False
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false', WANDB_MODE='offline')
assert (3,10) <= sys.version_info[:2] <= (3,12), 'Use a Python 3.10-3.12 runtime for this pinned environment'
SEEDS = [42, 1337, 2026]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Soter-vfinal')
REPO = ROOT / 'repo'
WORK = ROOT / 'runs'
assert (REPO / 'vfinal_pipeline/training.py').is_file(), 'Upload the execution package first'
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
# One writer per run. Drive has no cross-session distributed lock.


In [ ]:
if RUN_INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO/'configs/v-final/requirements-colab.txt')], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
    raise SystemExit('Restart the Colab runtime, set RUN_INSTALL=False, then rerun from the top')
# No install occurs with the default flag. Pin drift must be resolved, never ignored.
import importlib.metadata as md
for line in (REPO/'configs/v-final/requirements-colab.txt').read_text().splitlines():
    name, version = line.split('==')
    assert md.version(name) == version, (name, md.version(name), version)
import torch
assert torch.cuda.is_available(), 'GPU required for genuine training'
print(torch.cuda.get_device_name(0))


In [ ]:
# Prefer a user-staged local model. A mutable repository name is NOT a revision pin.
STUDENT_DIR = ROOT/'models/student-base'
STUDENT_REPO = 'sentence-transformers/all-MiniLM-L6-v2'
STUDENT_REVISION = ''  # Fill an approved exact 40-character commit SHA after license review.
if RUN_DOWNLOAD:
    import re
    assert re.fullmatch('[a-f0-9]{40}', STUDENT_REVISION), 'Exact approved revision required'
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=STUDENT_REPO, revision=STUDENT_REVISION, local_dir=str(STUDENT_DIR),
                      allow_patterns=['*.json','*.txt','*.safetensors','pytorch_model.bin'])
# All subsequent model access is local-only. Never execute remote modeling code.
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', HF_DATASETS_OFFLINE='1')


In [ ]:
from vfinal_pipeline.common import decode, canonical, write_new
from vfinal_pipeline.data_build import build, load_manifest
from vfinal_pipeline.experiment import run, mine, select_teacher, bundle_identity, resources
INPUT = ROOT/'accepted-input.json'
DATA = WORK/'accepted-data-v1'
spec = decode(INPUT.read_bytes())
assert spec['accepted'] is True and spec['exposure_audit_complete'] is True
assert all(s.get('privacy_review') == 'accepted' for s in spec['sources'])
assert any(s['role']=='exposure' for s in spec['sources']), 'Declare all teacher/baseline exposure; do not omit it'
if DATA.exists():
    d = decode((DATA/'manifest.json').read_bytes())
    data = load_manifest(DATA, d['identity'])
else:
    data = build(INPUT, DATA)
print(data['counts'], data['identity'])
from vfinal_pipeline.common import digest
assert data['input_manifest_sha256'] == digest(INPUT.read_bytes()), 'Accepted input changed: build a new data version'
# Holdout is sequestered by the builder; no training/mining/calibration cell loads it.


In [ ]:
cfg = decode((REPO/'configs/v-final/experiment.template.json').read_bytes())
cfg.update(data_dir=str(DATA), data_identity=data['identity'], base_dir=str(STUDENT_DIR), base_identity=bundle_identity(STUDENT_DIR))
cfg['training']['checkpoint_every'] = 1000
resources(WORK, False)
# Optional accepted pretrained teacher candidates: each needs matching 14-label order
# and complete declared training/selection exposure decontaminated against holdout.
TEACHER_CANDIDATES = []  # [{'path': str(ROOT/'models/teacher'), 'identity': bundle_identity(...)}]
if RUN_TEACHER_TRAIN:
    assert TEACHER_CANDIDATES, 'Supply accepted teacher initialization'
    tc = copy.deepcopy(cfg)
    tc.update(base_dir=TEACHER_CANDIDATES[0]['path'], base_identity=TEACHER_CANDIDATES[0]['identity'])
    tc['training']['alpha'] = 0.0
    teacher_result = run(tc, WORK/'teacher-finetune')
    TEACHER_CANDIDATES = [{'path': str(WORK/'teacher-finetune/best-model'), 'identity': teacher_result['model_identity']}]
if TEACHER_CANDIDATES:
    choice = select_teacher(cfg, TEACHER_CANDIDATES)
    cfg.update(teacher_dir=choice['selected']['path'], teacher_identity=choice['selected']['identity'])
    cfg['training']['alpha'] = 0.5
    from vfinal_pipeline.checkpoint import atomic
    atomic(WORK/'teacher-choice.json', canonical(choice))


In [ ]:
# Auto-resume is identity-bound: same config/data/model/teacher bytes + saved optimizer,
# scaler, RNG, epoch, optimizer step and batch cursor. Corruption raises, never silently restarts.
RUNS = []
if RUN_TRAIN:
    assert cfg['teacher_dir'], 'Distilled candidate requires an accepted selected teacher; use no_distillation ablation separately'
    for seed in SEEDS:
        c = copy.deepcopy(cfg)
        c['training']['seed'] = seed
        start = time.perf_counter()
        initial = WORK/f'seed-{seed}/initial'
        run(c, initial)
        mining = mine(c, initial/'best-model', split='train')
        c['hard_negative_ids'] = [r['id'] for r in mining['mistakes']]
        c.update(base_dir=str(initial/'best-model'), base_identity=bundle_identity(initial/'best-model'))
        final = WORK/f'seed-{seed}/hard-negatives'
        result = run(c, final)
        RUNS.append({'seed': seed, 'path': str(final), 'result': result, 'elapsed_seconds': time.perf_counter()-start})
    from vfinal_pipeline.checkpoint import atomic
    atomic(WORK/'seed-runs.json', canonical(RUNS))


In [ ]:
# Same data split and 3 seeds. These are actual training paths, not predicted ablation gains.
ABLATIONS = []
if RUN_ABLATIONS:
    assert cfg['teacher_dir']
    for variant in ['no_distillation','no_class_balance','no_hard_negatives']:
        for seed in SEEDS:
            c = copy.deepcopy(cfg)
            c['training']['seed'] = seed
            if variant == 'no_distillation': c['training']['alpha'] = 0.0
            if variant == 'no_class_balance': c['training']['class_balance'] = False
            out = WORK/f'ablations/{variant}/{seed}'
            result = run(c, out/'initial')
            if variant != 'no_hard_negatives':
                found = mine(c, out/'initial/best-model')
                c['hard_negative_ids'] = [r['id'] for r in found['mistakes']]
                c.update(base_dir=str(out/'initial/best-model'), base_identity=bundle_identity(out/'initial/best-model'))
                result = run(c, out/'hard-negatives')
            ABLATIONS.append({'variant':variant,'seed':seed,'result':result})
    from vfinal_pipeline.checkpoint import atomic
    atomic(WORK/'ablation-training.json', canonical(ABLATIONS))
# No-calibration and fp32-vs-int8 are evaluation/export ablations below, not retraining.
# Model soup is deliberately not adopted: no demonstrated aligned basin or validation gain.


In [ ]:
import statistics
if (WORK/'seed-runs.json').exists():
    observed = decode((WORK/'seed-runs.json').read_bytes())
    losses = [min(e['validation_nll'] for e in r['result']['history']) for r in observed]
    print({'scope':'validation selection only; NOT final comparison', 'n_seeds':len(losses),
           'validation_nll_mean':statistics.mean(losses), 'validation_nll_std':statistics.stdev(losses) if len(losses)>1 else None})
    # Use measured walltime and actual checkpoint bytes for session/storage planning.
    print({'checkpoint_bytes':sum(p.stat().st_size for p in WORK.rglob('state.bin')),
           'run_seconds':[r['elapsed_seconds'] for r in observed]})
# Select the final seed using validation NLL only, before opening any frozen holdout.


In [ ]:
if RUN_EXPORT:
    observed = decode((WORK/'seed-runs.json').read_bytes())
    selected = min(observed, key=lambda r:min(e['validation_nll'] for e in r['result']['history']))
    chosen = Path(selected['path'])
    export_spec = {'model_dir':str(chosen/'best-model'), 'calibration':str(chosen/'calibration.json'),
                   'max_length':cfg['max_length'], 'unit_only':False, 'quantize':True}
    EXPORT = WORK/'selected-export-v1'
    spec_path = WORK/'export-input.json'
    from vfinal_pipeline.checkpoint import atomic
    atomic(spec_path,canonical(export_spec))
    subprocess.run([sys.executable,'-m','vfinal_pipeline','export',str(spec_path),str(EXPORT)],check=True)
    # ONNX embeds temperature exactly once. Threshold is validation-selected metadata.
    # Repeat export with calibration temperature=1 only in a separately identified no-calibration ablation,
    # recomputing calibration identity and validation-only threshold. Never compare mutable artifacts.


## Identical-context final comparison — NOT RUN
Freeze gate configuration before scoring. Run the existing eval/real-serving harness on v14, v17-multilingual, both explicitly identified other versions, all three candidate seeds, and ablations. Export per-case risk probabilities aligned to exactly the same Case JSONL order; retain native multiclass confusion separately. Capture CPU/GPU batch-one and realistic concurrency, cold/warm, model/tokenizer/policy, memory/complete external weight size and external network separately. The gate recomputes all named slice/language metrics, grouped paired bootstrap intervals and McNemar statistics; under-supported slices cannot pass.
Historical receipts are not fresh inference. Fabricated JSON or UNIT-ONLY evidence never authorizes deployment. Offline `release_ready` is always false; real signed serving-path review and product-policy fixes are still mandatory. Populate `comparison-input.json` as documented in RUNBOOK from real outputs, not hand-written success flags.

In [ ]:
if RUN_GATE:
    comparison = ROOT/'comparison-input.json'
    subprocess.run([sys.executable,'-m','vfinal_pipeline','gate',str(comparison),str(WORK/'release-gate.json')],check=True)
# Exit 2 means incomplete/failed criteria; inspect the written report, do not weaken it.


In [ ]:
if RUN_PACKAGE:
    import shutil
    EXPORT = WORK/'selected-export-v1'
    shutil.copyfile(REPO/'docs/ml/v-final-offline/execution/MODEL_CARD.md', EXPORT/'MODEL_CARD.md')
    # Amend this new model card with actual training/data/export/gate identities before packaging.
    gate_path = WORK/'release-gate.json'
    assert gate_path.is_file(), 'Preserve a genuine gate report, even if failed; never invent a pass'
    package_spec = WORK/'package-input.json'
    from vfinal_pipeline.checkpoint import atomic
    atomic(package_spec,canonical({'source':str(EXPORT),'gate':str(gate_path)}))
    subprocess.run([sys.executable,'-m','vfinal_pipeline','package',str(package_spec),str(WORK/'quarantine-bundle-v1')],check=True)
    subprocess.run([sys.executable,'-m','vfinal_pipeline','verify',str(WORK/'quarantine-bundle-v1')],check=True)
# Never replace production models, signing policy, n8n nodes or IDE packages in this notebook.
